# R4-N1 WavLM-A: calibration, bundle, replay and handoff (full run on Colab)
This notebook runs the frozen recipe `configs/r4_n1_wavlm_calibration_a.json` from scratch with **transformers 5.22.0**: the tests, then audit, fit, operating, package, a two-tier replay and the handoff record. The recipe is the B WavLM procedure applied unchanged to Protocol A (team-agreed, user-reported 15 November 2026). It creates a **new** run folder under `runs/r4/r4_n1_wavlm/` and a new bundle `bundles/r4_n1_wavlm_head_A_calibrated`; nothing earlier is continued or overwritten. It trains nothing and reads no A test recording.

Owner: Subhan. Sync first (same paths on Drive): the code, tests, recipe and notebook; the exports `reports/r4/r4_u1_wavlm_A/`; the source bundle `bundles/r4_u1_wavlm_head_A/`; `data/manifests/cohort_v1.parquet`, `data/splits/protocol_A_v1.parquet`; the shards `data/processed/cohort_v1/`; and `configs/r3_n1_wavlm_calibration_b.json` (the recipe guard compares against it).

The operating role is opened **once**, after the mapping is frozen. If a stage fails after that point, do not rerun it in the same folder: read the error and ask first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Pin transformers
Colab ships a newer patch version. Run this cell, then **restart the session** (Runtime, Restart session) and continue from the next cell. The check cell fails if the version is wrong.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.22.0'], check=True)
import importlib.metadata as md
print('transformers', md.version('transformers'))

In [ ]:
from pathlib import Path
import hashlib, io, json, os, re, subprocess, sys, time
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Change if your project is elsewhere.
OWNER = 'Subhan'
SHARDS = ROOT / 'data/processed/cohort_v1'
BUNDLE = ROOT / 'bundles/r4_n1_wavlm_head_A_calibrated'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, str(ROOT / 'src'))
import torch, transformers, numpy, scipy, pyarrow
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| numpy', numpy.__version__, '| scipy', scipy.__version__, '| pyarrow', pyarrow.__version__)
assert transformers.__version__ == '5.22.0', 'The pin is transformers 5.22.0: install it and restart the session'
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()
required = ['configs/r4_n1_wavlm_calibration_a.json', 'configs/r3_n1_wavlm_calibration_b.json', 'scripts/r4_n1_wavlm_calibrate.py', 'scripts/r4_n1_wavlm_replay.py',
            'src/echoproof/evaluation/r4_n1_wavlm.py', 'src/echoproof/models/r4_n1_wavlm_bundle.py',
            'src/echoproof/evaluation/r3_n1_wavlm.py', 'src/echoproof/evaluation/r3_n1_wavlm_operating.py', 'src/echoproof/models/r3_n1_wavlm_bundle.py',
            'reports/r4/r4_u1_wavlm_A/selection.json', 'reports/r4/r4_u1_wavlm_A/evidence.json', 'bundles/r4_u1_wavlm_head_A/bundle.json',
            'data/manifests/cohort_v1.parquet', 'data/splits/protocol_A_v1.parquet']
missing = [p for p in required if not (ROOT / p).is_file()]
assert not missing, f'Missing: {missing}'
assert SHARDS.is_dir(), 'Set SHARDS to the canonical shard folder'
assert not BUNDLE.exists(), f'{BUNDLE} already exists: the packager never overwrites. Rename or remove the old bundle on Drive and rerun this cell.'
print('inputs present; the calibrated bundle folder is free')

## Tests (nothing is fitted until they pass)
The CPU tests for the A pipeline and bundle, the replay comparison tests, and the real-library adapter tests (a tiny random WavLM through transformers 5.22.0). A failure or a skip stops the notebook.

In [ ]:
tests = ['tests/test_r4_n1_wavlm.py', 'tests/test_r3_n1_wavlm_replay.py', 'tests/test_r3_u1_adapter.py', 'tests/test_r2_bundle.py']
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', '-rs', *tests], text=True, capture_output=True)
print(check.stdout[-3500:], check.stderr[-1500:])
skips = re.findall(r'^SKIPPED \[\d+\] (.+)$', check.stdout, flags=re.M)
assert check.returncode == 0, 'Resolve the test failures first'
assert not skips, f'Unexpected skips (a missing dependency?): {skips}'
print('tests passed; skips: none')

## Stage A: audit
Creates the new run folder, freezes a copy of the recipe, verifies the saved exports against the recipe, `evidence.json` and the locked A cohort and split manifests (the test partition is withheld). The run folder is parsed from the output.

In [ ]:
def stage(command, *extra):
    result = subprocess.run([sys.executable, 'scripts/r4_n1_wavlm_calibrate.py', command, '--root', str(ROOT), '--owner', OWNER, *extra], text=True, capture_output=True)
    print(result.stdout[-3000:], result.stderr[-2000:])
    assert result.returncode == 0, f'{command} failed'
    return result.stdout
out = stage('audit')
RUN = Path(re.search(r'audit passed: (.+)', out).group(1).strip())
print('RUN =', RUN)

## Stage B: choose and fit the mapping (calibration role only)
Leave-one-speaker-out comparison of identity, temperature and affine, the pre-declared selection rule, the final fit and the freeze. The operating file is not opened. Note: WavLM-A makes 3 errors in 1,200 calibration recordings, so the fit is weakly determined; whichever mapping the rule selects is reported as it is.

In [ ]:
stage('fit', '--run-dir', str(RUN))

## Stage C: the operating role, opened once
Acceptance rule, threshold with its plateau report, rows A to D, per-component panels, reliability, speaker diagnostics, bootstrap. A second attempt in the same run is refused.

In [ ]:
stage('operating', '--run-dir', str(RUN))

## Stage D: package the calibrated bundle

In [ ]:
stage('package', '--run-dir', str(RUN))

## Replay (transformers 5.22.0)
All operating recordings and 100 each from the selection and calibration roles, in two tiers (the bundle's scorer on canonical windows, and WAV bytes through `analyze`). Tolerance on recording scores 1e-3; a flip is explained only when the expected score is within the tolerance of the threshold. The A bundle is loaded with `load_wavlm_bundle_a` (the shared `load_inference_bundle` routes `wavlm_head` to the B loader, which refuses A).

In [ ]:
subprocess.run([sys.executable, 'scripts/r4_n1_wavlm_replay.py', '--root', str(ROOT), '--run-dir', str(RUN), '--shard-dir', str(SHARDS), '--owner', OWNER,
                '--device', 'cuda' if torch.cuda.is_available() else 'cpu'], check=True)

## Handoff record
Writes `handoff/handoff.json` with the bundle, run and evidence hashes. It never overwrites.

In [ ]:
from echoproof.artifacts import write_json
replay = json.loads((RUN / 'handoff/replay.json').read_text(encoding='utf-8'))
assert replay['passed'] is True, 'The replay did not pass'
fit = json.loads((RUN / 'fit/calibration.json').read_text(encoding='utf-8'))
op = json.loads((RUN / 'operating/threshold.json').read_text(encoding='utf-8'))
metrics = json.loads((RUN / 'operating/metrics.json').read_text(encoding='utf-8'))
print('selected mapping:', fit['selected_mapping'], fit['selected_parameters'], '| LOSO means:', fit['selection']['means'])
print('accepted:', op['calibration_accepted'], '|', op['acceptance']['reason'], '| score type:', op['score_type'])
print('rule:', op['decision_rule'], '| plateau:', op['plateau'])
for k, v in metrics['rows'].items():
    f = v['full']
    print(f"{k:34s} F1 {f['macro_f1']:.4f}  FPR {f['genuine_fpr']:.4f}  recall {f['tts_recall']:.4f}  TP {f['tp']} FP {f['fp']} TN {f['tn']} FN {f['fn']}")
handoff = {'task': 'R4-N1-WavLM handoff', 'owner': OWNER, 'run_dir': RUN.name, 'bundle': str(BUNDLE), 'bundle_json_sha256': sha256(BUNDLE / 'bundle.json'),
           'calibration_component_sha256': sha256(BUNDLE / 'calibration/calibration.json'), 'recipe_sha256': sha256(ROOT / 'configs/r4_n1_wavlm_calibration_a.json'),
           'fit_calibration_sha256': sha256(RUN / 'fit/calibration.json'), 'operating_evidence_sha256': sha256(RUN / 'operating/evidence.json'),
           'replay_sha256': sha256(RUN / 'handoff/replay.json'), 'selected_mapping': fit['selected_mapping'], 'calibration_accepted': op['calibration_accepted'],
           'score_type': op['score_type'], 'decision_rule': op['decision_rule'], 'replay_passed': replay['passed'], 'decision_flips': len(replay['flips']),
           'a_test_read': False, 'locks_changed': False,
           'environment': {'transformers': transformers.__version__, 'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}}
write_json(RUN / 'handoff/handoff.json', handoff)
print('handoff written:', RUN / 'handoff/handoff.json')